# Notebook 02 — Preprocessing
## AI-Powered Credit Risk Analyzer
**Dataset:** German Credit Risk  
**Integrante 1:** Datos + EDA + Preprocessing  
**Objetivo:** Limpiar, codificar y dividir el dataset para entrenamiento.

## 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
import joblib

import warnings
warnings.filterwarnings('ignore')

# Crear carpetas de salida si no existen
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/processed/plots', exist_ok=True)

print('Librerías cargadas ✓')

Librerías cargadas ✓


## 2. Cargar dataset

In [2]:
df = pd.read_csv('../data/raw/german_credit_data.csv', index_col=0)
print(f'Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas')
df.head()

Dataset cargado: 1000 filas, 10 columnas


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


## 3. Limpieza de datos

In [3]:
# 3.1 Rellenar nulos en variables categóricas con 'unknown'
# Estos nulos tienen significado informativo (probablemente = sin cuenta)
df['Saving accounts'] = df['Saving accounts'].fillna('unknown')
df['Checking account'] = df['Checking account'].fillna('unknown')

print('Nulos restantes:')
print(df.isnull().sum())
print('\n✓ Sin valores nulos')

Nulos restantes:
Age                 0
Sex                 0
Job                 0
Housing             0
Saving accounts     0
Checking account    0
Credit amount       0
Duration            0
Purpose             0
Risk                0
dtype: int64

✓ Sin valores nulos


In [4]:
# 3.2 Verificar duplicados
dups = df.duplicated().sum()
print(f'Duplicados: {dups}')
# Si hubiera duplicados: df.drop_duplicates(inplace=True)

Duplicados: 0


## 4. Encoding de la variable objetivo

In [5]:
# Convertir Risk: good=0, bad=1 (convencion estandar: 1=evento de interes=default)
df['Risk'] = df['Risk'].map({'good': 0, 'bad': 1})

print('Encoding de Risk:')
print(df['Risk'].value_counts())
print('\n0 = good (no default) | 1 = bad (default)')

Encoding de Risk:
Risk
0    700
1    300
Name: count, dtype: int64

0 = good (no default) | 1 = bad (default)


## 5. Separar features y target

In [6]:
X = df.drop(columns=['Risk'])
y = df['Risk']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'Columnas: {list(X.columns)}')

Features (X): (1000, 9)
Target (y): (1000,)
Columnas: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose']


## 6. Train / Validation / Test split

In [7]:
# Split estratificado para mantener la proporción de clases en cada subset
# División: 70% train | 15% val | 15% test

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Tamaños de los splits:')
print(f'  Train:      {X_train.shape[0]} filas ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Validation: {X_val.shape[0]} filas ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'  Test:       {X_test.shape[0]} filas ({X_test.shape[0]/len(X)*100:.1f}%)')

print('\nProporción de clases por split (% bad=1):')
print(f'  Train:      {y_train.mean()*100:.1f}%')
print(f'  Validation: {y_val.mean()*100:.1f}%')
print(f'  Test:       {y_test.mean()*100:.1f}%')
print('\n✓ stratify=y mantiene la proporcion en todos los splits')

Tamaños de los splits:
  Train:      700 filas (70.0%)
  Validation: 150 filas (15.0%)
  Test:       150 filas (15.0%)

Proporción de clases por split (% bad=1):
  Train:      30.0%
  Validation: 30.0%
  Test:       30.0%

✓ stratify=y mantiene la proporcion en todos los splits


## 7. Encoding de variables categóricas

In [8]:
# Definir columnas por tipo
num_cols = ['Age', 'Credit amount', 'Duration']

# Variables ordinales: tienen orden natural
ordinal_cols = ['Saving accounts', 'Checking account']
ordinal_categories = [
    ['unknown', 'little', 'moderate', 'quite rich', 'rich'],  # Saving accounts
    ['unknown', 'little', 'moderate', 'rich']                  # Checking account
]

# Variables nominales: sin orden
nominal_cols = ['Sex', 'Housing', 'Purpose']

# Job ya es numérico (0-3)
print('Tipos de columnas definidos:')
print(f'  Numéricas:  {num_cols}')
print(f'  Ordinales:  {ordinal_cols}')
print(f'  Nominales:  {nominal_cols}')
print(f'  Numérica ya codificada: ["Job"]')

Tipos de columnas definidos:
  Numéricas:  ['Age', 'Credit amount', 'Duration']
  Ordinales:  ['Saving accounts', 'Checking account']
  Nominales:  ['Sex', 'Housing', 'Purpose']
  Numérica ya codificada: ["Job"]


In [9]:
# 7.1 Encoding ordinal
# IMPORTANTE: ajustar (fit) solo sobre train, luego transform sobre val y test

ord_enc = OrdinalEncoder(
    categories=ordinal_categories,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_train[ordinal_cols] = ord_enc.fit_transform(X_train[ordinal_cols])
X_val[ordinal_cols]   = ord_enc.transform(X_val[ordinal_cols])
X_test[ordinal_cols]  = ord_enc.transform(X_test[ordinal_cols])

print('Categorías aprendidas por OrdinalEncoder:')
for col, cats in zip(ordinal_cols, ord_enc.categories_):
    print(f'  {col}: {list(cats)}')

Categorías aprendidas por OrdinalEncoder:
  Saving accounts: ['unknown', 'little', 'moderate', 'quite rich', 'rich']
  Checking account: ['unknown', 'little', 'moderate', 'rich']


In [10]:
# 7.2 Encoding nominal (One-Hot Encoding)
X_train = pd.get_dummies(X_train, columns=nominal_cols, drop_first=False)
X_val   = pd.get_dummies(X_val,   columns=nominal_cols, drop_first=False)
X_test  = pd.get_dummies(X_test,  columns=nominal_cols, drop_first=False)

# Alinear columnas (por si alguna categoria no aparece en val/test)
X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f'Columnas después de OHE: {X_train.shape[1]}')
print(list(X_train.columns))

Columnas después de OHE: 19
['Age', 'Job', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Sex_female', 'Sex_male', 'Housing_free', 'Housing_own', 'Housing_rent', 'Purpose_business', 'Purpose_car', 'Purpose_domestic appliances', 'Purpose_education', 'Purpose_furniture/equipment', 'Purpose_radio/TV', 'Purpose_repairs', 'Purpose_vacation/others']


## 8. Escalado de variables numéricas

In [11]:
# StandardScaler: media=0, std=1
# IMPORTANTE: fit solo sobre train para evitar data leakage

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols]   = scaler.transform(X_val[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print('Escalado aplicado (StandardScaler):')
print(f'  Media aprendida:  {scaler.mean_.round(3)}')
print(f'  Std aprendida:    {scaler.scale_.round(3)}')
print(f'\n✓ fit solo sobre train → sin data leakage')

Escalado aplicado (StandardScaler):
  Media aprendida:  [  35.469 3200.873   20.769]
  Std aprendida:    [  10.946 2673.031   11.954]

✓ fit solo sobre train → sin data leakage


## 9. Verificación final

In [12]:
print('=== Resumen del preprocessing ===' )
print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}   | y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}  | y_test:  {y_test.shape}')
print(f'\nNulos restantes en X_train: {X_train.isnull().sum().sum()}')
print(f'Nulos restantes en X_val:   {X_val.isnull().sum().sum()}')
print(f'Nulos restantes en X_test:  {X_test.isnull().sum().sum()}')
print('\n✓ Listo para modelado')

X_train.head(3)

=== Resumen del preprocessing ===
X_train: (700, 19) | y_train: (700,)
X_val:   (150, 19)   | y_val:   (150,)
X_test:  (150, 19)  | y_test:  (150,)

Nulos restantes en X_train: 0
Nulos restantes en X_val:   0
Nulos restantes en X_test:  0

✓ Listo para modelado


,Age,Job,Saving accounts,Checking account,Credit amount,Duration,Sex_female,Sex_male,Housing_free,Housing_own,Housing_rent,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others
10,-0.956361,2,1.0,2.0,-0.713001,-0.733512,True,False,False,False,True,False,True,False,False,False,False,False,False
82,-1.047717,1,2.0,0.0,-0.610869,-0.231598,True,False,False,False,True,True,False,False,False,False,False,False,False
827,0.048549,2,1.0,0.0,0.360687,-0.231598,False,True,False,True,False,True,False,False,False,False,False,False,False


## 10. Exportar datos procesados y artefactos

In [14]:
# Guardar splits
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv',     index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)

y_train.to_csv('../data/processed/y_train.csv', index=False, header=True)
y_val.to_csv('../data/processed/y_val.csv',     index=False, header=True)
y_test.to_csv('../data/processed/y_test.csv',   index=False, header=True)

# Guardar artefactos del preprocessing (necesarios para la app)
joblib.dump(scaler,  '../data/processed/scaler.pkl')
joblib.dump(ord_enc, '../data/processed/ordinal_encoder.pkl')

# Guardar lista de columnas finales (importante para la app de Streamlit)
import json
with open('../data/processed/feature_columns.json', 'w') as f:
    json.dump(list(X_train.columns), f, indent=2)

print('Archivos exportados a data/processed/:')
import os
for f in sorted(os.listdir('../data/processed/')):
    if not os.path.isdir(f'../data/processed/{f}'):
        size = os.path.getsize(f'../data/processed/{f}')
        print(f'  {f} ({size/1024:.1f} KB)')

Archivos exportados a data/processed/:
  .gitkeep (0.0 KB)
  X_test.csv (21.4 KB)
  X_train.csv (98.9 KB)
  X_val.csv (20.8 KB)
  feature_columns.json (0.4 KB)
  ordinal_encoder.pkl (1.3 KB)
  scaler.pkl (0.9 KB)
  y_test.csv (0.3 KB)
  y_train.csv (1.4 KB)
  y_val.csv (0.3 KB)


## 11. Nota sobre desbalance de clases

El dataset presenta un desbalance de clases: **70% good (0) y 30% bad (1)**. Este aspecto debe tenerse en cuenta durante el modelado. Existen dos estrategias principales para manejarlo:

**Opción A — `class_weight='balanced'`** (recomendada para empezar, no requiere modificar los datos):
```python
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced')
```

**Opción B — SMOTE** (oversampling sintético, modifica el train set):
```python
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
```

> ⚠️ Si se usa SMOTE, aplicarlo **solo sobre el train set**, nunca sobre val o test.